In [1]:
import os
os.environ["NPU_VISIBLE_DEVICES"]="7"
os.environ["ASCEND_RT_VISIBLE_DEVICES"]="7"
import json
from typing import Dict, List, Any
from tqdm import tqdm
from functools import partial

import math
import numpy as np
import matplotlib.pyplot as plt

import torch
import torch_npu
from transformers import AutoModelForCausalLM, AutoTokenizer
from datasets import load_dataset

/home/lihz/miniconda3/envs/workspace/lib/python3.10/site-packages/torch_npu/utils/path_manager.py:82: UserWarning: Warning: The /usr/local/Ascend/ascend-toolkit/latest owner does not match the current user.
  warnings.warn(f"Warning: The {path} owner does not match the current user.")
/home/lihz/miniconda3/envs/workspace/lib/python3.10/site-packages/torch_npu/utils/path_manager.py:82: UserWarning: Warning: The /usr/local/Ascend/ascend-toolkit/8.0.RC2/aarch64-linux/ascend_toolkit_install.info owner does not match the current user.
  warnings.warn(f"Warning: The {path} owner does not match the current user.")


In [2]:
base_model = "/data/pretrained-models/meta/Llama-3.2-3B-Instruct"
tokenizer = AutoTokenizer.from_pretrained(base_model)

In [3]:
base_dataset = "/data/datasets/Llama-3.2-3B-Instruct-evals"
general_datasets = [
    "/data/datasets/ultrachat_200k",
]
reason_datasets = [
    "Llama-3.2-3B-Instruct-evals__mmlu__details",
    "Llama-3.2-3B-Instruct-evals__gpqa__details",
    "Llama-3.2-3B-Instruct-evals__arc_challenge__details"
]
math_datasets = [
    "gsm8k",
    "/data/datasets/MathInstruct",
    "Llama-3.2-3B-Instruct-evals__math__details"
]
# humaneval_datasets = [
#     "evalplus/humanevalplus",
# ]
# mbpp_datasets = [
#     "evalplus/mbppplus"
# ]
magicoder_datasets = [
    "/data/datasets/Magicoder-Evol-Instruct-110K",
]
leetcode_datasets = [
    "greengerong/leetcode"
]

In [4]:
def preprocess_gsm8k(examples:Dict[str, Any], tokenizer:AutoTokenizer)->Dict[str, Any]:
    prefix = "Given the following problem, reason and give a final answer to the problem.\nProblem: {{question}}\nYour response should end with \"The final answer is [answer]\" where [answer] is the response to the problem.\n"
    icl = [
        {
            "role" : "user",
            "content" : "There are 15 trees in the grove. Grove workers will plant trees in the grove today. After they are done, there will be 21 trees. How many trees did the grove workers plant today?"
        },
        {
            "role" : "assistant",
            "content" : "There are 15 trees originally. Then there were 21 trees after some more were planted. So there must have been 21 - 15 = 6. The final answer is 6"
        },
        {
            "role": "user",
            "content": "If there are 3 cars in the parking lot and 2 more cars arrive, how many cars are in the parking lot?"
        },
        {
            "role": "assistant",
            "content" : "There are originally 3 cars. 2 more cars arrive. 3 + 2 = 5. The final answer is 5"
        },
        {
            "role": "user",
            "content" : "Leah had 32 chocolates and her sister had 42. If they ate 35, how many pieces do they have left in total?",
        },
        {
            "role" : "assistant",
            "content" : "Originally, Leah had 32 chocolates. Her sister had 42. So in total they had 32 + 42 = 74. After eating 35, they had 74 - 35 = 39. The final answer is 39"
        },
        {
            "role" : "user",
            "content" : "Jason had 20 lollipops. He gave Denny some lollipops. Now Jason has 12 lollipops. How many lollipops did Jason give to Denny?"
        },
        {
            "role" : "assistant",
            "content" : "Jason started with 20 lollipops. Then he had 12 after giving some to Denny. So he gave Denny 20 - 12 = 8. The final answer is 8"
        },
        {
            "role" : "user",
            "content" : "Shawn has five toys. For Christmas, he got two toys each from his mom and dad. How many toys does he have now?"
        },
        {
            "role" : "assistant",
            "content" : "Shawn started with 5 toys. If he got 2 toys each from his mom and dad, then that is 4 more toys. 5 + 4 = 9. The final answer is 9"
        },
        {
            "role" : "user",
            "content" : "There were nine computers in the server room. Five more computers were installed each day, from monday to thursday. How many computers are now in the server room?"
        },
        {
            "role" : "assistant",
            "content" : "There were originally 9 computers. For each of 4 days, 5 more computers were added. So 5 * 4 = 20 computers were added. 9 + 20 is 29. The final answer is 29"
        },
        {
            "role" : "user",
            "content" : "Michael had 58 golf balls. On tuesday, he lost 23 golf balls. On wednesday, he lost 2 more. How many golf balls did he have at the end of wednesday?"
        },
        {
            "role" : "assistant",
            "content" : "Michael started with 58 golf balls. After losing 23 on tuesday, he had 58 - 23 = 35. After losing 2 more, he had 35 - 2 = 33 golf balls. The final answer is 33"
        },
        {
            "role" : "user",
            "content" : "Olivia has $23. She bought five bagels for $3 each. How much money does she have left?"
        },
        {
            "role" : "assistant",
            "content" : "Olivia had 23 dollars. 5 bagels for 3 dollars each will be 5 x 3 = 15 dollars. So she has 23 - 15 dollars left. 23 - 15 is 8. The final answer is 8"
        }
    ]
    for i in range(len(icl)):
        if icl[i]['role'] == "user":
            icl[i]['content'] = prefix.replace("{{question}}", icl[i]['content'])
    history = []
    for i in range(len(icl)):
        if i % 2 == 0:
            history.append([icl[i]['content']])
        else:
            history[-1].append(icl[i]['content'])
    return {
        "inst": prefix.replace("{{question}}", examples["question"].strip()),
        "history": history, 
        "response": examples["answer"].strip()
        }

def preprocess_llama3_eval(examples:Dict[str, Any], tokenizer:AutoTokenizer) -> Dict[str, Any]:
    input_final_prompts = examples['input_final_prompts'][0].replace("<|eot_id|>", "")
    input_final_prompts = input_final_prompts.replace("<|start_header_id|>user<|end_header_id|>", "#$%2$%$#")
    input_final_prompts = input_final_prompts.split("#$%2$%$#")[1:]
    input_final_prompts = [item.split("<|start_header_id|>assistant<|end_header_id|>") for item in input_final_prompts]
    # history = []
    # for i in range(len(input_final_prompts[:-1])):
    #     if i % 2 == 0:
    #         history.append([input_final_prompts[i]])
    #     else:
    #         history[-1].append(input_final_prompts[i])
    return {
        "inst": input_final_prompts[-1][0],
        "response": examples['output_prediction_text'][0],
        # "history": history if history != 0 else [],
        "history": input_final_prompts[:-1]
    }

def preprocess_ultrachat(examples:Dict[str, Any], tokenizer:AutoTokenizer)->Dict[str,Any]:
    messages = examples['messages']
    history = []
    for i in range(len(messages)):
        if i % 2 == 0:
            history.append([messages[i]['content']])
        else:
            history[-1].append(messages[i]['content'])
    if len(history[-1]) < 2:
        print(examples['messages'])
    return { 
        "inst": history[-1][0],
        "response": history[-1][1],
        "history": history[:-1] if len(history[:-1]) != 0 else []
    }

In [5]:

def task_preprocess(example:Dict[str, str], tokenizer:AutoTokenizer, task:str="humaneval")->Dict[str, str]:
    if task == "humaneval":
        instruction_prefix = "Please provide a self-contained Python script that solves the following problem in a markdown code block:"
        response_prefix = "Below is a Python script with a self-contained function that solves the problem and passes corresponding tests:"
        # some random words which servcleaes as the splitter
        _MAGIC_SPLITTER_ = "-[[]]-this-is-really-our-highest-priority-[[]]-"
        task_prompt = f"""\
{instruction_prefix}
```
{example['prompt'].strip()}
```
"""
        response = f"""\
{response_prefix}
```python
{example}
```
"""
        task_prompt = tokenizer.apply_chat_template(
            [
                {"role": "user", "content": task_prompt},
                {"role": "assistant", "content": response},
            ],
            tokenize=False,
        ).split(_MAGIC_SPLITTER_)[0]
        return {
            "inst": task_prompt,
        }
    elif task == "mbpp":
        instruction_prefix = "Please provide a self-contained Python script that solves the following problem in a markdown code block:"
        response_prefix = "Below is a Python script with a self-contained function that solves the problem and passes corresponding tests:"
        # some random words which servcleaes as the splitter
        _MAGIC_SPLITTER_ = "-[[]]-this-is-really-our-highest-priority-[[]]-"
#         task_prompt = f"""\
# {example['prompt'].strip()}
# {example['test_list'][0].strip()}
# {instruction_prefix}
# ```python
# ......
# ```
# """
#         response = f"""\
# {response_prefix}
# ```python
# {_MAGIC_SPLITTER_}
# ```
# """
        python_prefix = 'Write a python function to '
        func_prefix = 'Write a function to '
        if python_prefix in example['prompt']:
            prefix = python_prefix
        elif func_prefix in example['prompt']:
            prefix = func_prefix
        else:
            prefix = ""
        prompt = example['prompt'].replace(prefix, '').strip().capitalize()
        task_prompt = f"""\
{instruction_prefix}
```
{example['code'].split(":")[0].strip()}:
    \"\"\"
    {prompt}
    >>> {example['test_list'][0].replace("assert", "").strip()}
    True
    \"\"\"
```
"""
        response = f"""\
{response_prefix}
```python
{_MAGIC_SPLITTER_}
```
"""
        task_prompt = tokenizer.apply_chat_template(
            [
                {"role": "user", "content": task_prompt},
                {"role": "assistant", "content": response},
            ],
            tokenize=False,
        ).split(_MAGIC_SPLITTER_)[0]
        return {
            "inst": task_prompt,
        }
    elif task == "magicoder":
        return {
            "inst": example['instruction'].strip(),
            "response": example['response'].strip(),
            "history": [],
        }
    elif task == "mathinstruct":
        return {
            "inst": example['instruction'].strip(),
            "response": example['output'].strip(),
            "history": [],
        }
    elif task == "leetcode":
        return {
            "inst": example["content"].strip(),
            "response": example["python"].strip(),
            "history": [],
        }
  

In [6]:
dataset = load_dataset(
    base_dataset,
    name="Llama-3.2-3B-Instruct-evals__math__details",
    num_proc=8,
)['latest']

In [7]:
print(dataset[1065]['input_final_prompts'][0])

 this step-by-step format:

## Step 1: [Concise description]
[Brief explanation and calculations]

## Step 2: [Concise description]
[Brief explanation and calculations]

...

Regardless of the approach, always conclude with:

Therefore, the final answer is: $\boxed{answer}$. I hope it is correct.

Where [answer] is just the final number or expression that solves the problem.

Problem: Find the domain of the expression $\frac{\sqrt{x-2}}{\sqrt{5-x}}$.<|eot_id|><|start_header_id|>assistant<|end_header_id|>

## Step 1: Consider the expression inside the first square root
The expression inside the first square root is $x-2$, and it must be non-negative. This means that $x-2 \ge 0$, which simplifies to $x \ge 2$.

## Step 2: Consider the expression inside the second square root
The expression inside the second square root is $5-x$, and it must be non-negative. This means that $5-x \ge 0$, which simplifies to $x \le 5$.

## Step 3: Consider the denominator of the expression
The denominator o

In [8]:
general_datasets = [
    load_dataset(
        general_datasets[0],
        num_proc=8,
    )['train_sft'],
]
general_datasets = [
    general_datasets[0].filter(
        lambda x: len(x['messages']) > 1,
    ).map(
        partial(preprocess_ultrachat, tokenizer=tokenizer),
        num_proc=8
    )
]
reason_datasets = [
    load_dataset(
        base_dataset,
        name=item,
        num_proc=8,
    )['latest'].map(
        partial(preprocess_llama3_eval, tokenizer=tokenizer),
        num_proc=8,
    ) for item  in reason_datasets
]
math_datasets = [
    load_dataset(
        math_datasets[0],
        "main",
        num_proc=8,
    )['train'].map(
        partial(preprocess_gsm8k, tokenizer=tokenizer),
        num_proc=8,
    ),
    load_dataset(
        math_datasets[1],
        num_proc=8,
    ).map(
        partial(task_preprocess, tokenizer=tokenizer, task="mathinstruct"),
        num_proc=8,
    )['train'],
    load_dataset(
        base_dataset,
        name=math_datasets[2],
        num_proc=8,
    )['latest'].map(
        partial(preprocess_llama3_eval, tokenizer=tokenizer),
        num_proc=8,
    )
]

magicoder_datasets = [
    load_dataset(
        item,
        num_proc=8,
    ).map(
        partial(task_preprocess, tokenizer=tokenizer, task="magicoder"),
        num_proc=8,
        load_from_cache_file=False,
    )['train'] for item in magicoder_datasets
]
leetcode_datasets = [
    load_dataset(
        item,
        num_proc=8,
    ).map(
        partial(task_preprocess, tokenizer=tokenizer, task="leetcode"),
        num_proc=8,
    )['train'] for item in leetcode_datasets
]

Map (num_proc=8):   0%|          | 0/14042 [00:00<?, ? examples/s]

Map (num_proc=8):   0%|          | 0/448 [00:00<?, ? examples/s]

Map (num_proc=8):   0%|          | 0/1165 [00:00<?, ? examples/s]

Map (num_proc=8):   0%|          | 0/5000 [00:00<?, ? examples/s]

Map (num_proc=8):   0%|          | 0/111183 [00:00<?, ? examples/s]

In [9]:
# print(magicoder_datasets[0][4]['inst'])
# print(mbpp_datasets[0][5]['inst'])
# print(mbpp_datasets[0][5]['code'])

In [10]:
# print(humaneval_datasets[0][4]['inst'])
# print(humaneval_datasets[0][1]['canonical_solution'])

In [11]:
print(leetcode_datasets[0])

Dataset({
    features: ['id', 'slug', 'title', 'difficulty', 'content', 'java', 'c++', 'python', 'javascript', 'inst', 'response', 'history'],
    num_rows: 2360
})


In [12]:
# print(general_datasets[2][0]['input_final_prompts'][0])
print(general_datasets[0][200]['inst'])
# print(general_datasets[0][200]['history'][-1])

That was a great horror story! Can you add more foreshadowing and suspense to the beginning to make the ending even more shocking?


In [13]:
# print(math_datasets[0][0]['input_final_prompts'][0])
# print(math_datasets[0][0]['input_correct_responses'])
# print(math_datasets[-1][0]['input_final_prompts'][0])
# print(math_datasets[-1][0]['history'][0][0])
print(math_datasets[-1][1065]['inst'])



Solve the following math problem efficiently and clearly:

- For simple problems (2 steps or fewer):
Provide a concise solution with minimal explanation.

- For complex problems (3 steps or more):
Use this step-by-step format:

## Step 1: [Concise description]
[Brief explanation and calculations]

## Step 2: [Concise description]
[Brief explanation and calculations]

...

Regardless of the approach, always conclude with:

Therefore, the final answer is: $\boxed{answer}$. I hope it is correct.

Where [answer] is just the final number or expression that solves the problem.

Problem: On the game board below Kendra will start at the center of the board. For each turn she will spin this spinner with four congruent sectors once, and then she will move one space in the direction indicated on the spinner. The "Start'' square does not have a numerical value, but Kendra may land on it during her turns. What is the probability that the sum of the numbers in the spaces on which she will land wil

In [14]:
print(math_datasets[1][1]['inst'])
print(math_datasets[1][1]['output'])

How many ways can the letters in the word COMMON be arranged?
Answer Choices: (A) 6 (B) 30 (C) 90 (D) 120 (E) 180
Let's solve the multi-choice question step by step.
According to the above the # of permutations of 6 letters COMMON out of which 2 O's and 2 M's are identical is 6!2!∗2!=180
The answer is E.


In [15]:
# print(magicoder_datasets[0][3]['inst'])
# print(magicoder_datasets[0][3].keys())
print(magicoder_datasets[0][3]['response'])

This task requires writing of a significant volume of code, which is not fully suitable for a text-based medium. However, I will outline a general solution using Python and scikit-learn. We'll use "CountVectorizer" for bag-of-words model and "TfidVectorizer" for TF-IDF. To handle different languages, we can use 'langdetect' library.

1. Import required libraries
```python
import pandas as pd
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import classification_report, accuracy_score, confusion_matrix
from langdetect import detect
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
from nltk.tokenize import word_tokenize
import nltk
nltk.download('punkt')
nltk.download('wordnet')
nltk.download('stopwords')
```

2. Load sentence data and labels. For example, if data is stored in a csv format:
```python
data = pd.read_cs

In [16]:
print(len(general_datasets[0]),
      len(reason_datasets[0]),
      len(reason_datasets[1]),
      len(math_datasets[0]), 
      len(math_datasets[1]),
      len(math_datasets[2]),
      len(leetcode_datasets[0]), 
      len(magicoder_datasets[0]))

207865 14042 448 7473 262039 5000 2360 111183


In [17]:
mix_domains = []

samples = 2000

repeat = 1 if len(general_datasets[0]) >= samples else 2
for _ in range(repeat):
    for i, item in enumerate(general_datasets[0]):
        if i == samples:
            break
        mix_domains.append({
            'inputs': item['inst'],
            'task': 'ultrachat',
            'task_label': 0,
            'label': 0,
            'outputs': item['response'],
            'history': item['history'],
        })

In [18]:
repeat = 1 if len(reason_datasets[0]) >= samples else 2
for _ in range(repeat):
    for i, item in enumerate(reason_datasets[0]):
        if i == 1200:
            break
        mix_domains.append({
            'inputs': item['inst'],
            'task': 'mmlu',
            'task_label': 1,
            'label': 1,
            'outputs': item['response'],
            'history': item['history']
        })



In [19]:
repeat = 1 if len(math_datasets[0]) >= samples else 2
for _ in range(repeat):
    for i, item in enumerate(math_datasets[0]):
        if i == samples:
            break
        mix_domains.append({
            'inputs': item['inst'],
            'task': 'gsm8k',
            'task_label': 4,
            'label': 2,
            'outputs': item['response'],
            'history': item['history']
        })

repeat = 1 if len(math_datasets[2]) >= samples else 2
for _ in range(repeat):
    for i, item in enumerate(math_datasets[2]):
        if i == samples:
            break
        mix_domains.append({
            'inputs': item['inst'],
            'task': 'math',
            'task_label': 6,
            'label': 2,
            'outputs': item['response'],
            'history': item['history']
        })

In [20]:
# with open("/data/lihz/datasets/mix_domains/mix_domains_48_v2_part1.jsonl", 'w') as f:
#     for item in mix_domains:
#         f.write(json.dumps(item) + '\n')

In [21]:
# mix_domains = []

repeat = 1 if len(reason_datasets[1]) >= samples else 2
for _ in range(repeat):
    for i, item in enumerate(reason_datasets[1]):
        if i == 400:
            break
        mix_domains.append({
            'inputs': item['inst'],
            'task': 'gpqa',
            'task_label': 2,
            'label': 1,
            'outputs': item['response'],
            'history': item['history']
        })

repeat = 1 if len(reason_datasets[2]) >= samples else 2
for _ in range(repeat):
    for i, item in enumerate(reason_datasets[2]):
        if i == 1200:
            break
        mix_domains.append({
            'inputs': item['inst'],
            'task': 'arc_c',
            'task_label': 3,
            'label': 1,
            'outputs': item['response'],
            'history': item['history']
        })

repeat = 1 if len(math_datasets[1]) >= samples else 2
for _ in range(repeat):
    for i, item in enumerate(math_datasets[1]):
        if i == samples:
            break
        mix_domains.append({
            'inputs': item['inst'],
            'task': 'mathinstruct',
            'task_label': 5,
            'label': 2,
            'outputs': item['response'],
            'history': item['history']
        })

repeat = 1 if len(leetcode_datasets[0]) >= samples else 2
for _ in range(repeat):
    for i, item in enumerate(leetcode_datasets[0]):
        if i == 1000:
            break
        mix_domains.append({
            'inputs': item['inst'],
            'task': 'leetcode',
            'task_label': 7,
            'label': 3,
            'outputs': item['response'],
            'history': item['history']
        })

repeat = 1 if len(magicoder_datasets[0]) >= samples else 2
for _ in range(repeat):
    for i, item in enumerate(magicoder_datasets[0]):
        if i == samples:
            break
        mix_domains.append({
            'inputs': item['inst'],
            'task': 'magicoder',
            'task_label': 8,
            'label': 3,
            'outputs': item['response'],
            'history': item['history']
        })

In [22]:
with open("/data/lihz/datasets/mix_domains/mix_domains_48_v2.jsonl", 'w') as f:
    for item in mix_domains:
        f.write(json.dumps(item) + '\n')

In [23]:
import json
with open("/data/lihz/datasets/mix_domains/mix_domains_48_v2.jsonl", 'r') as f:
    for i, line in enumerate(f):
        # if isinstance(json.loads(line)['history'], str):
        #     print(json.loads(line)['history'])
        # print(json.loads(line)['inputs'])
        # print(json.loads(line)['task'])
        # print(json.loads(line)['task_label'])
        # print(json.loads(line)['label'])
        # print(json.loads(line)['outputs'])
        # print(len(json.loads(line)['history']))
        # if i == 33:
            # break
        # if not isinstance(json.loads(line)['history'][0], list):
        #     print(json.loads(line)['history'][0])
        #     print(i)
        #     break
        if json.loads(line)['history'] is None:
            print(i)
            break
        if json.loads(line)['history'] is not None:
            if len(json.loads(line)['history']) > 0:
                # if len(json.loads(line)['history'][0]) < 2:
                #     print(i)
                #     break
                for item in json.loads(line)['history']:
                    if len(item) < 2:
                        print(item)
                        print(json.loads(line)['inputs'])
                        # print(json.loads(line)['history'][0][1])
                        # print(json.loads(line)['history'][1][1])
                        # print(json.loads(line)['outputs'])
                        print(i)
                        assert False

with open("/data/lihz/datasets/mix_domains_eval/mix_domains_eval_v9.jsonl", 'r') as f:
    for i, line in enumerate(f):
        if json.loads(line)['history'] is None:
            print(i)
            break

In [24]:
from datasets import load_dataset
json_dataset = load_dataset(
    "json",
    data_files=["/data/lihz/datasets/mix_domains/mix_domains_48_v2.jsonl"],
)
# json_dataset = load_dataset(
#     "json",
#     data_files=["/data/lihz/datasets/mix_domains/mix_domains_48_v2_part2.jsonl"],
# )

Generating train split: 0 examples [00:00, ? examples/s]

In [30]:
import os
# del os.environ['HF_ENDPOINT']
# os.environ['http_proxy']="127.0.0.1:9150"
# os.environ['https_proxy']="127.0.0.1:9150"
from datasets import load_dataset
math_dataset = load_dataset("EleutherAI/hendrycks_math", 'geometry')
print(os.environ['http_proxy'])
# 'algebra', 'counting_and_probability', 'geometry', 'intermediate_algebra', 'number_theory', 'prealgebra', 'precalculus'
print(type(math_dataset), math_dataset.keys(), len(math_dataset['train']), len(math_dataset['test']))
print(math_dataset['train'].column_names)
print(math_dataset['train'][0]['problem'])
print(math_dataset['train'][0]['solution'])

127.0.0.1:9150
<class 'datasets.dataset_dict.DatasetDict'> dict_keys(['train', 'test']) 870 479
['problem', 'level', 'type', 'solution']
Square ABCD has its center at $(8,-8)$ and has an area of 4 square units. The top side of the square is horizontal. The square is then dilated with the dilation center at (0,0) and a scale factor of 2. What are the coordinates of the vertex of the image of square ABCD that is farthest from the origin? Give your answer as an ordered pair.
With the center of dilation at the origin and a scale factor of 2, all the coordinates of square $ABCD$ are twice the coordinates of its preimage. The preimage has an area of 4 square units, so its side length is 2 units. Since the center of the preimage is at $(8, -8)$, the four vertices of the preimage are at $(7, -9), (7, -7), (9, -7)$ and $(9, -9)$. The point $(9, -9)$ is the farthest from the origin on the preimage, so the point farthest from the origin on the image of square $ABCD$ is $\boxed{(18, -18)}.$


In [ ]:
math_dataset = load_dataset("meta-llama/Llama-3.2-3B-Instruct-evals", 'Llama-3.2-3B-Instruct-evals__math__details')
# print(len(math_dataset['latest'][0]['input_final_prompts'][0].split("<|start_header_id|>user<|end_header_id|>")[1:]))
# print(math_dataset['latest'][0]['input_final_prompts'][0].split("<|start_header_id|>user<|end_header_id|>")[1:][0])
# print(math_dataset['latest'][1002]['input_final_prompts'][0])


In [17]:
cosmos_dataset = load_dataset("allenai/cosmos_qa")
print(type(cosmos_dataset), cosmos_dataset.keys(), len(cosmos_dataset['train']))
print(cosmos_dataset['train'].column_names)

datasets/allenai/cosmos_qa@main/cosmos_qa.py /data/lihz/.cache/huggingface/datasets/downloads/bb18864e7112c51e2e0d155435f7ddc04fe05d3281bb584ae4a856b38114fec2.py.incomplete


datasets/allenai/cosmos_qa@main/README.md /data/lihz/.cache/huggingface/datasets/downloads/3870b8b71e313807cbbe08ecd41ace481084f52b3cb72213fba19565bca9cc4f.incomplete


Generating train split:   0%|          | 0/25262 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/6963 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/2985 [00:00<?, ? examples/s]

In [26]:
square_dataset = load_dataset("rajpurkar/squad_v2")
print(type(square_dataset), square_dataset.keys(), len(square_dataset['train']))
print(square_dataset['train'].column_names)

<class 'datasets.dataset_dict.DatasetDict'> dict_keys(['train', 'validation']) 130319
['id', 'title', 'context', 'question', 'answers']
